# Task 2: Classificatore SVM Lineare su Feature Linguistiche Non Lessicali

Questo notebook implementa un sistema di classificazione basato su caratteristiche linguistiche avanzate (estratte tramite strumenti di NLP e basate sullo schema *Universal Dependencies*). A differenza del Task 1, l'addestramento della **Support Vector Machine (SVM)** non si basa sulle parole o sui caratteri (lessico), bensì su feature stilometriche, morfosintattiche e di complessità sintattica (*Profiling Linguistico*).

In [12]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Disattiviamo i warning di frammentazione di Pandas
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

## 1. Caricamento dei Dataset di Supporto

Prima di elaborare i vettori di profili linguistici, carichiamo i dati originali preprocessati per recuperare i testi e le etichette reali (`label`).

In [13]:
df_train = pd.read_pickle("../data/processed/df_train_processed.pkl")
df_val = pd.read_pickle("../data/processed/df_val_processed.pkl")
df_test = pd.read_pickle("../data/processed/df_test_processed.pkl")

## 2. Caricamento e Allineamento dei Profili Linguistici (UD Features)

I file CSV contengono centinaia di feature non lessicali associate ai documenti. Estraiamo il `doc_id` dal nome del file per ordinare i record e garantire un perfetto allineamento posizionale con le etichette dei dataset originari.

In [14]:
def load_and_align_features(csv_path, target_columns=None):
    """Carica le feature linguistiche, estrae l'ID documento e lo ordina per l'allineamento."""
    df = pd.read_csv(csv_path, delimiter='\t')
    
    # Creazione della colonna doc_id ed estrazione dell'indice numerico
    doc_ids = df['Filename'].str.extract(r'(\d+)')[0].astype(int)
    df.insert(0, 'doc_id', doc_ids)
    
    # Ordinamento e impostazione dell'indice per garantire la coerenza
    df = df.sort_values(by='doc_id').set_index('doc_id')
    X = df.drop(columns=['Filename'])
    
    # Se specificato, riallinea le colonne (indispensabile per Validation e Test)
    if target_columns is not None:
        X = X.reindex(columns=target_columns, fill_value=0)
        
    return X

# 1. Elaborazione TRAINING set
X_train_ud = load_and_align_features('../data/processed/data_task2/train_UD.csv')
y_train_ud = df_train['label'].copy()

# 2. Elaborazione VALIDATION set (con allineamento colonne sul Train)
X_val_ud = load_and_align_features('../data/processed/data_task2/validation_UD.csv', target_columns=X_train_ud.columns)
y_val_ud = df_val['label'].copy()

# 3. Elaborazione TEST set (con allineamento colonne sul Train)
X_test_ud = load_and_align_features('../data/processed/data_task2/test_UD.csv', target_columns=X_train_ud.columns)
y_test_ud = df_test['label'].copy()

print(f"Feature estratte per il Train: {X_train_ud.shape[1]}")

Feature estratte per il Train: 141


## 3. Definizione della Pipeline e Valutazione delle Performance

Poiché le feature linguistiche hanno scale e unità di misura molto eterogenee (es. conteggi assoluti vs densità lessicale), utilizziamo uno **StandardScaler** prima di alimentare il classificatore **LinearSVC**. 
Valutiamo il comportamento del modello sia sul Validation Set che sul Test Set indipendente.

In [15]:
# Definizione della pipeline di modellazione
model_ud = Pipeline([
    ("scaler", StandardScaler()),  
    ("clf", LinearSVC(random_state=42, max_iter=10000))  
])

# Addestramento sui dati del profilo linguistico
model_ud.fit(X_train_ud, y_train_ud)

# Predizioni e metriche
pred_val_ud = model_ud.predict(X_val_ud)
pred_test_ud = model_ud.predict(X_test_ud)

print("=== VALIDATION SET REPORT (Profiling-UD) ===")
print(classification_report(y_val_ud, pred_val_ud, zero_division=0))

print("\n=== TEST SET REPORT (Profiling-UD) ===")
print(classification_report(y_test_ud, pred_test_ud, zero_division=0))

=== VALIDATION SET REPORT (Profiling-UD) ===
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       500
           1       0.94      0.94      0.94       500

    accuracy                           0.94      1000
   macro avg       0.94      0.94      0.94      1000
weighted avg       0.94      0.94      0.94      1000


=== TEST SET REPORT (Profiling-UD) ===
              precision    recall  f1-score   support

           0       0.84      0.93      0.88       500
           1       0.92      0.82      0.87       500

    accuracy                           0.88      1000
   macro avg       0.88      0.88      0.88      1000
weighted avg       0.88      0.88      0.88      1000



## 4. Analisi dell'Incertezza: i 5 Casi più Ambigui

Sfruttando la funzione `decision_function` di SVM, calcoliamo la distanza geometrica dei campioni del Test Set dall'iperpiano di decisione. Minore è la distanza in valore assoluto, maggiore è l'incertezza del modello sul documento.

In [16]:
# Calcolo delle distanze dal confine di decisione
distanze = model_ud.decision_function(X_test_ud)

# Costruzione del DataFrame per il tracciamento dell'incertezza
df_incertezza = pd.DataFrame({
    'distanza_assoluta': np.abs(distanze),
    'distanza_reale': distanze,
    'testo': df_test['text'],
    'label_reale': y_test_ud,
    'predizione': pred_test_ud
})

# Isola le 5 istanze più vicine allo zero (i casi più ambigui)
i_5_più_incerti = df_incertezza.sort_values(by='distanza_assoluta').head(5)

# Visualizzazione pulita dei casi critici
i_5_più_incerti[['distanza_reale', 'label_reale', 'predizione', 'testo']]

,distanza_reale,label_reale,predizione,testo
411,-0.006241,1,0,"Una qualificazione che, francamente, lascia un..."
539,-0.007560,0,0,"Culle di pelliccia, quindi, amiche del respiro..."
620,0.014183,1,1,"Una cifra considerevole, che si somma a quelle..."
666,-0.033602,1,0,"According to Amnesty, sino al 2023 sarebbero s..."
60,0.034147,0,1,Ma il trauma rimane. A prendere di mira la tur...


In [17]:
for row in i_5_più_incerti[['testo', 'label_reale']].itertuples(index=False):
    print(f'{row.label_reale}: {row.testo}')

1: Una qualificazione che, francamente, lascia un po’ di amaro in bocca. Non per la prestazione del Feyenoord, squadra solida e mai doma, ma per quella dei giallorossi. Si avvertiva la tensione, il peso di un match decisivo, e la Roma, pur giocando tra le mura amiche, ha faticato a imporre il proprio gioco. Troppo compassati, a tratti rinchiusi nella propria metà campo, i ragazzi di De Rossi hanno concesso fin troppo spazio agli olandesi, rischiando più di quanto vorrebbe la prudenza. Un rigore trasformato da Paulo Dybala, un lampo di genio in una notte grigia, ha fatto la differenza, ma senza quella sterilità offensiva che pure si è vista in più occasioni. Qualcuno parlerà di cinismo, di gestione del vantaggio, di esperienza nel saper soffrire. Ma noi crediamo che ci sia altro da sottolineare. Questa Roma ha bisogno di maggiore personalità, di un’identità più definita, di un coraggio che vada oltre il risultato. Si è visto un passo avanti rispetto al passato, certo, l’impronta di De R

## 5. Interpretabilità del Modello: Feature Dominanti

Un grande vantaggio delle SVM con kernel lineare è l'ispezionabilità dei coefficienti (`coef_`). I pesi positivi spingono la classificazione verso la classe target (1), mentre i pesi negativi verso la classe di controllo (0). Vediamo le **20 caratteristiche linguistiche più determinanti** in valore assoluto.

In [18]:
# Estrazione dei coefficienti assegnati dal classificatore lineare
pesi = model_ud['clf'].coef_[0]

df_feature_importanti = pd.DataFrame({
    'Feature': X_train_ud.columns,
    'Peso': pesi,
    'Peso_Assoluto': np.abs(pesi)
})

# Ordinamento in base all'importanza globale (valore assoluto)
le_20_dominanti = df_feature_importanti.sort_values(by='Peso_Assoluto', ascending=False).head(20)
le_20_dominanti[['Feature', 'Peso']].reset_index(drop=True)

,Feature,Peso
0,n_tokens,-2.419553
1,dep_dist_advmod,1.874903
2,upos_dist_ADV,-1.525584
3,lexical_density,-1.457214
4,n_prepositional_chains,1.309277
5,ttr_form_chunks_200,-1.297201
6,dep_dist_det,-1.232503
7,upos_dist_NUM,-1.121213
8,upos_dist_PUNCT,1.016801
9,upos_dist_PRON,-0.999782


### 💡 Analisi dei Pattern Linguistici Rilevati

Dall'estrazione delle feature dominanti emergono considerazioni interessanti sulla struttura dei testi:
* **Feature con peso negativo importante (es. `n_tokens`, `upos_dist_ADV`, `lexical_density`)**: Indicano che i testi più lunghi, con una forte presenza di avverbi e un'alta densità lessicale, sono associati dal modello alla **Classe 0**.
* **Feature con peso positivo importante (es. `dep_dist_advmod`, `n_prepositional_chains`)**: Pattern legati a costrutti sintattici più stratificati o modificatori avverbiali guidano il modello verso l'assegnazione della **Classe 1**.

Rispetto al Task 1 basato su n-grammi di caratteri, questo modello mostra una stabilità decisamente superiore e una minor perdita di performance nel passaggio dal Validation al Test set (da **0.95** a **0.88** di F1-macro), provando l'efficacia e la robustezza delle feature non lessicali contro i rischi di domain shift.